# Image Segmentation and Edge Detection

This notebook processes images from `images/train` step by step:
1. Threshold-based mask creation
2. Gradient-based edge detection
3. Batch mask generation for the whole training set
4. Preview of an original image and its mask

## Setup

In [ ]:
saved_mask_path = mask_dir / f'{first_image_path.stem}_mask.png'
saved_edge_path = edge_dir / f'{first_image_path.stem}_edge.png'
saved_mask = np.array(Image.open(saved_mask_path))
saved_edge = np.array(Image.open(saved_edge_path))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(first_image)
axes[0].set_title(f'Original: {first_image_path.name}')
axes[0].axis('off')

axes[1].imshow(saved_mask, cmap='gray')
axes[1].set_title('Saved Mask')
axes[1].axis('off')

axes[2].imshow(saved_edge, cmap='gray')
axes[2].set_title('Saved Edge Image')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
saved_paths = []
saved_edge_paths = []

for image_path in image_files:
    image_array = load_image(image_path)
    gray_array = to_gray(image_array)
    mask_array = create_threshold_mask(gray_array, threshold_value)
    edge_array, _ = create_edge_image(gray_array, edge_threshold)
    output_path = mask_dir / f'{image_path.stem}_mask.png'
    edge_output_path = edge_dir / f'{image_path.stem}_edge.png'
    Image.fromarray(mask_array).save(output_path)
    Image.fromarray(edge_array).save(edge_output_path)
    saved_paths.append(output_path)
    saved_edge_paths.append(edge_output_path)

print(f'Saved {len(saved_paths)} masks to {mask_dir}')
print(f'Saved {len(saved_edge_paths)} edge images to {edge_dir}')
print('Example saved file:', saved_paths[0].name if saved_paths else 'None')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

axes[0].imshow(first_image)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(first_gray, cmap='gray')
axes[1].set_title('Grayscale')
axes[1].axis('off')

axes[2].imshow(first_mask, cmap='gray')
axes[2].set_title(f'Threshold Mask (t={threshold_value})')
axes[2].axis('off')

axes[3].imshow(first_magnitude, cmap='magma')
axes[3].set_title('Gradient Magnitude')
axes[3].axis('off')

axes[4].imshow(first_edge_mask, cmap='gray')
axes[4].set_title(f'Edge Mask (t={edge_threshold})')
axes[4].axis('off')

comparison = np.concatenate([first_mask, first_edge_mask], axis=1)
axes[5].imshow(comparison, cmap='gray')
axes[5].set_title('Mask | Edge Mask')
axes[5].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

train_dir = Path('/Users/muhammadjonparpiyev/Library/Mobile Documents/com~apple~CloudDocs/Documents/DIP/Week 10/Lab 2/images/train')
mask_dir = Path('/Users/muhammadjonparpiyev/Library/Mobile Documents/com~apple~CloudDocs/Documents/DIP/Week 10/Lab 2/images/output_masks')
example_dir = Path('/Users/muhammadjonparpiyev/Library/Mobile Documents/com~apple~CloudDocs/Documents/DIP/Week 10/Lab 2/images/output_examples')
edge_dir = Path('/Users/muhammadjonparpiyev/Library/Mobile Documents/com~apple~CloudDocs/Documents/DIP/Week 10/Lab 2/images/output_edges')
mask_dir.mkdir(parents=True, exist_ok=True)
example_dir.mkdir(parents=True, exist_ok=True)
edge_dir.mkdir(parents=True, exist_ok=True)

threshold_value = 128
edge_threshold = 0.12


def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    return np.array(image)


def to_gray(image_array):
    return np.array(Image.fromarray(image_array).convert('L'), dtype=np.float32)


def create_threshold_mask(gray_array, threshold=128):
    return (gray_array >= threshold).astype(np.uint8) * 255


def gradient_magnitude(gray_array):
    gray_norm = gray_array / 255.0
    gradient_y, gradient_x = np.gradient(gray_norm)
    magnitude = np.sqrt(gradient_x ** 2 + gradient_y ** 2)
    return magnitude


def create_edge_image(gray_array, threshold=edge_threshold):
    magnitude = gradient_magnitude(gray_array)
    return (magnitude >= threshold).astype(np.uint8) * 255, magnitude


image_files = sorted([path for path in train_dir.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
first_image_path = image_files[0]
first_image = load_image(first_image_path)
first_gray = to_gray(first_image)
first_mask = create_threshold_mask(first_gray, threshold_value)
first_edge_mask, first_magnitude = create_edge_image(first_gray, edge_threshold)

print(f'Loaded {first_image_path.name}')
print(f'Image shape: {first_image.shape}')
print(f'Gray range: {first_gray.min():.1f} to {first_gray.max():.1f}')

## 1-5. Single Image: Threshold Mask and Gradient Edges

## 6-9. Apply the Same Threshold to the Whole Training Set

## 10. Preview One Original Image and Its Saved Mask